# Dataset preparation


In [1]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

candidates = [
    Path("/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline"),
    Path("/content/drive/MyDrive/Sales-analytics-data-pipeline"),
]

PROJECT_ROOT = next(
    (p for p in candidates if (p / ".git").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. Check the folder containing your .git directory."
    )

print("Project root:", PROJECT_ROOT)

for folder in [
    "data/raw",
    "data/processed",
    "data/rejected",
    "logs",
    "src/sales_pipeline",
    "tests",
    "docs",
    "config",
]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Phase 1 directories are ready.")


Mounted at /content/drive
Project root: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
Phase 1 directories are ready.


In [3]:

from pathlib import Path
import zipfile
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline"
)

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {DRIVE_PROJECT}"
    )

print("Project folder:", DRIVE_PROJECT)
print("\nDataset files found:\n")

found_files = []

for path in DRIVE_PROJECT.rglob("*"):
    if not path.is_file():
        continue

    # Ignore Git's internal files
    if ".git" in path.parts:
        continue

    if path.suffix.lower() in {".csv", ".zip"}:
        found_files.append(path)
        print(
            f"{path.relative_to(DRIVE_PROJECT)} "
            f"({path.stat().st_size / (1024 * 1024):.2f} MB)"
        )

if not found_files:
    print(
        "\nNo CSV or ZIP files found inside the project folder. "
        "Check where the downloaded dataset was saved."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/Sales-analytics-data-pipeline

Dataset files found:

Sales-analytics-data-pipeline/data/raw/archive.zip (42.65 MB)


In [4]:

from pathlib import Path
import zipfile

ZIP_PATH = (
    DRIVE_PROJECT
    / "Sales-analytics-data-pipeline"
    / "data"
    / "raw"
    / "archive.zip"
)

# If the path above does not exist, locate the archive automatically.
if not ZIP_PATH.exists():
    matches = list(DRIVE_PROJECT.rglob("archive.zip"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one archive.zip; found {len(matches)}"
        )
    ZIP_PATH = matches[0]

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    files = [
        item for item in archive.infolist()
        if not item.is_dir()
    ]

    print("Archive:", ZIP_PATH)
    print("Files inside archive:", len(files))
    print("\nArchive contents:")

    for item in files:
        print(
            f"{item.filename} "
            f"({item.file_size / (1024 * 1024):.2f} MB)"
        )


Archive: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/raw/archive.zip
Files inside archive: 9

Archive contents:
olist_customers_dataset.csv (8.62 MB)
olist_geolocation_dataset.csv (58.44 MB)
olist_order_items_dataset.csv (14.72 MB)
olist_order_payments_dataset.csv (5.51 MB)
olist_order_reviews_dataset.csv (13.78 MB)
olist_orders_dataset.csv (16.84 MB)
olist_products_dataset.csv (2.27 MB)
olist_sellers_dataset.csv (0.17 MB)
product_category_name_translation.csv (0.00 MB)


In [5]:

from pathlib import Path
import zipfile

# Reuse DRIVE_PROJECT from your earlier Colab cell.
matches = list(DRIVE_PROJECT.rglob("archive.zip"))

if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected exactly one archive.zip; found {len(matches)}"
    )

ZIP_PATH = matches[0]
OLIST_DIR = ZIP_PATH.parent / "olist"
OLIST_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for item in archive.infolist():
        if item.is_dir() or not item.filename.lower().endswith(".csv"):
            continue

        if Path(item.filename).name != item.filename:
            raise ValueError(f"Unexpected archive path: {item.filename}")

        target = OLIST_DIR / item.filename

        if target.exists():
            print(f"Already extracted: {item.filename}")
            continue

        with archive.open(item) as source, target.open("wb") as output:
            while chunk := source.read(1024 * 1024):
                output.write(chunk)

print("\nExtraction complete.")
print("Dataset directory:", OLIST_DIR)
print("CSV files:", len(list(OLIST_DIR.glob("*.csv"))))

for file in sorted(OLIST_DIR.glob("*.csv")):
    print(f"{file.name}: {file.stat().st_size / (1024 * 1024):.2f} MB")



Extraction complete.
Dataset directory: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/raw/olist
CSV files: 9
olist_customers_dataset.csv: 8.62 MB
olist_geolocation_dataset.csv: 58.44 MB
olist_order_items_dataset.csv: 14.72 MB
olist_order_payments_dataset.csv: 5.51 MB
olist_order_reviews_dataset.csv: 13.78 MB
olist_orders_dataset.csv: 16.84 MB
olist_products_dataset.csv: 2.27 MB
olist_sellers_dataset.csv: 0.17 MB
product_category_name_translation.csv: 0.00 MB


In [6]:

import pandas as pd
from collections import Counter

PROFILE_DIR = OLIST_DIR
CHUNK_SIZE = 100_000

profile_results = {}

for file_path in sorted(PROFILE_DIR.glob("*.csv")):
    print("\n" + "=" * 75)
    print("FILE:", file_path.name)

    total_rows = 0
    null_counts = Counter()
    sample = None
    columns = None

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        encoding="utf-8"
    ):
        if columns is None:
            columns = chunk.columns.tolist()
            sample = chunk.head(3).copy()

        total_rows += len(chunk)
        null_counts.update(chunk.isna().sum().to_dict())

    profile_results[file_path.name] = {
        "rows": total_rows,
        "columns": columns,
        "null_counts": dict(null_counts),
    }

    print("Rows:", f"{total_rows:,}")
    print("Columns:", len(columns))
    print("Column names:", columns)
    print("\nSample records:")
    display(sample)

    missing = {
        col: count
        for col, count in null_counts.items()
        if count > 0
    }

    print("\nColumns with missing values:")
    if missing:
        for col, count in missing.items():
            print(f"  {col}: {count:,} ({count / total_rows:.2%})")
    else:
        print("  None")

print("\n" + "=" * 75)
print("DATA PROFILING COMPLETED")



FILE: olist_customers_dataset.csv
Rows: 99,441
Columns: 5
Column names: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Sample records:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



Columns with missing values:
  None

FILE: olist_geolocation_dataset.csv
Rows: 1,000,163
Columns: 5
Column names: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Sample records:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



Columns with missing values:
  None

FILE: olist_order_items_dataset.csv
Rows: 112,650
Columns: 7
Column names: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Sample records:


...


Columns with missing values:
  product_category_name: 610 (1.85%)
  product_name_lenght: 610 (1.85%)
  product_description_lenght: 610 (1.85%)
  product_photos_qty: 610 (1.85%)
  product_weight_g: 2 (0.01%)
  product_length_cm: 2 (0.01%)
  product_height_cm: 2 (0.01%)
  product_width_cm: 2 (0.01%)

FILE: olist_sellers_dataset.csv
Rows: 3,095
Columns: 4
Column names: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Sample records:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



Columns with missing values:
  None

FILE: product_category_name_translation.csv
Rows: 71
Columns: 2
Column names: ['product_category_name', 'product_category_name_english']

Sample records:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto



Columns with missing values:
  None

DATA PROFILING COMPLETED


Create *extract.py*


In [26]:

from pathlib import Path
import importlib
import sys

# Project paths
REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = REPO_ROOT / "data" / "raw" / "olist"
PACKAGE_DIR = REPO_ROOT / "src" / "sales_pipeline"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

(PACKAGE_DIR / "__init__.py").touch()

extract_code = r'''
from pathlib import Path
import pandas as pd


# Explicit mapping to the actual source filenames.
DATASET_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}


# Required source columns, based on our profiling results.
DATASET_SCHEMAS = {
    "orders": [
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "order_id", "order_item_id", "product_id",
        "seller_id", "shipping_limit_date",
        "price", "freight_value",
    ],
    "customers": [
        "customer_id", "customer_unique_id",
        "customer_zip_code_prefix", "customer_city",
        "customer_state",
    ],
    "products": [
        "product_id", "product_category_name",
        "product_name_lenght", "product_description_lenght",
        "product_photos_qty", "product_weight_g",
        "product_length_cm", "product_height_cm",
        "product_width_cm",
    ],
    "sellers": [
        "seller_id", "seller_zip_code_prefix",
        "seller_city", "seller_state",
    ],
    "payments": [
        "order_id", "payment_sequential", "payment_type",
        "payment_installments", "payment_value",
    ],
    "reviews": [
        "review_id", "order_id", "review_score",
        "review_comment_title", "review_comment_message",
        "review_creation_date", "review_answer_timestamp",
    ],
    "geolocation": [
        "geolocation_zip_code_prefix", "geolocation_lat",
        "geolocation_lng", "geolocation_city",
        "geolocation_state",
    ],
    "category_translation": [
        "product_category_name",
        "product_category_name_english",
    ],
}


def load_csv(data_dir, dataset_name, chunksize=None):
    """
    Load an Olist CSV after validating its required columns.

    Args:
        data_dir: Directory containing the source CSV files.
        dataset_name: Key from DATASET_FILES.
        chunksize: Optional number of rows per chunk.

    Returns:
        A pandas DataFrame, or a pandas TextFileReader if chunked.
    """
    if dataset_name not in DATASET_FILES:
        raise ValueError(
            f"Unknown dataset '{dataset_name}'. "
            f"Available datasets: {list(DATASET_FILES)}"
        )

    path = Path(data_dir) / DATASET_FILES[dataset_name]

    if not path.is_file():
        raise FileNotFoundError(
            f"Dataset file not found: {path}"
        )

    # Read only the header first to verify the schema.
    header = pd.read_csv(path, nrows=0, encoding="utf-8")

    required_columns = DATASET_SCHEMAS[dataset_name]
    missing_columns = [
        column for column in required_columns
        if column not in header.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Schema mismatch in {path.name}. "
            f"Missing required columns: {missing_columns}"
        )

    return pd.read_csv(
        path,
        chunksize=chunksize,
        encoding="utf-8",
        low_memory=False,
    )
'''

extract_path = PACKAGE_DIR / "extract.py"
extract_path.write_text(
    extract_code.lstrip(),
    encoding="utf-8",
)

# Clear the imported module so the saved version is used.
sys.modules.pop("src.sales_pipeline.extract", None)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.extract as extract_module

print("Extraction module saved:", extract_path)
print("Imported module:", extract_module.__file__)
print("Payment mapping:", extract_module.DATASET_FILES["payments"])


Extraction module saved: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/extract.py
Imported module: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/extract.py
Payment mapping: olist_order_payments_dataset.csv


In [27]:

import pandas as pd

from src.sales_pipeline.extract import (
    load_csv,
    DATASET_FILES,
    DATASET_SCHEMAS,
)

# Expected row counts from our profiling
EXPECTED_ROWS = {
    "orders": 99_441,
    "order_items": 112_650,
    "customers": 99_441,
    "products": 32_951,
    "sellers": 3_095,
    "payments": 103_886,
    "reviews": 99_224,
    "geolocation": 1_000_163,
    "category_translation": 71,
}

results = []

for name in DATASET_FILES:
    df = load_csv(DATA_DIR, name)

    assert len(df) == EXPECTED_ROWS[name], (
        f"{name}: expected {EXPECTED_ROWS[name]}, got {len(df)}"
    )

    assert list(DATASET_SCHEMAS[name]) == [
        col for col in df.columns
        if col in DATASET_SCHEMAS[name]
    ], f"Schema verification failed for {name}"

    results.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "status": "PASS",
    })

results_df = pd.DataFrame(results)
display(results_df)

assert (results_df["status"] == "PASS").all()
print("All nine datasets passed extraction tests.")


,dataset,rows,columns,status
0,orders,99441,8,PASS
1,order_items,112650,7,PASS
2,customers,99441,5,PASS
3,products,32951,9,PASS
4,sellers,3095,4,PASS
5,payments,103886,5,PASS
6,reviews,99224,7,PASS
7,geolocation,1000163,5,PASS
8,category_translation,71,2,PASS


All nine datasets passed extraction tests.


In [28]:
#Save in github

%cd /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline

!git status --short
!git check-ignore data/raw/archive.zip data/raw/olist/olist_orders_dataset.csv


/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
Refresh index: 100% (14/14), done.
?? 01_data_ingestion.ipynb
?? src/sales_pipeline/extract.py
?? src/sales_pipeline/validate.py
data/raw/archive.zip
data/raw/olist/olist_orders_dataset.csv


In [29]:

from pathlib import Path
import sys
import importlib
import pandas as pd

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)
DATA_DIR = REPO_ROOT / "data" / "raw" / "olist"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.extract as extract_module
importlib.reload(extract_module)

load_csv = extract_module.load_csv

EXPECTED_ROWS = {
    "orders": 99_441,
    "order_items": 112_650,
    "customers": 99_441,
    "products": 32_951,
    "sellers": 3_095,
    "payments": 103_886,
    "reviews": 99_224,
    "geolocation": 1_000_163,
    "category_translation": 71,
}

results = []

for name, filename in extract_module.DATASET_FILES.items():
    if name == "geolocation":
        reader = load_csv(DATA_DIR, name, chunksize=100_000)
        row_count = sum(len(chunk) for chunk in reader)
        column_count = len(extract_module.DATASET_SCHEMAS[name])
    else:
        df = load_csv(DATA_DIR, name)
        row_count = len(df)
        column_count = len(df.columns)

    status = "PASS" if row_count == EXPECTED_ROWS[name] else "FAIL"

    results.append({
        "dataset": name,
        "rows": row_count,
        "columns": column_count,
        "status": status,
    })

results_df = pd.DataFrame(results)
display(results_df)

assert (results_df["status"] == "PASS").all(), "Extraction test failed."
print("SUCCESS: All nine datasets passed.")


,dataset,rows,columns,status
0,orders,99441,8,PASS
1,order_items,112650,7,PASS
2,customers,99441,5,PASS
3,products,32951,9,PASS
4,sellers,3095,4,PASS
5,payments,103886,5,PASS
6,reviews,99224,7,PASS
7,geolocation,1000163,5,PASS
8,category_translation,71,2,PASS


SUCCESS: All nine datasets passed.


In [31]:

%cd /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline

!git status --short
!git check-ignore -v data/raw/archive.zip data/raw/olist/olist_orders_dataset.csv
!git ls-files data/raw

/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
?? 01_data_ingestion.ipynb
?? src/sales_pipeline/extract.py
?? src/sales_pipeline/validate.py
.gitignore:12:data/*	data/raw/archive.zip
.gitignore:12:data/*	data/raw/olist/olist_orders_dataset.csv


Create *validate.py*

In [32]:

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

VALIDATE_FILE = REPO_ROOT / "src/sales_pipeline/validate.py"
VALIDATE_FILE.parent.mkdir(parents=True, exist_ok=True)

code = r'''
"""Data quality validation for the Olist sales analytics pipeline."""

from pathlib import Path

import pandas as pd

from .extract import load_csv


def _record(results, check_name, passed, details):
    """Append one validation result."""
    results.append({
        "check": check_name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details),
    })


def _check_not_null(df, column, label, results):
    missing = int(df[column].isna().sum())
    _record(
        results,
        f"{label}: {column} has no nulls",
        missing == 0,
        f"Missing values: {missing}",
    )


def _check_unique(df, columns, label, results):
    duplicates = int(df.duplicated(subset=columns).sum())
    _record(
        results,
        f"{label}: {', '.join(columns)} is unique",
        duplicates == 0,
        f"Duplicate rows for key: {duplicates}",
    )


def _check_foreign_key(
    child_df, child_column, parent_df, parent_column, label, results
):
    child_values = child_df[child_column].dropna()
    parent_values = parent_df[parent_column].dropna()
    unmatched = int((~child_values.isin(parent_values)).sum())

    _record(
        results,
        label,
        unmatched == 0,
        f"Unmatched non-null values: {unmatched}",
    )


def _check_non_negative(df, column, label, results):
    values = pd.to_numeric(df[column], errors="coerce")
    invalid = int((values < 0).sum())
    unparseable = int((values.isna() & df[column].notna()).sum())

    _record(
        results,
        f"{label}: {column} is non-negative",
        invalid == 0 and unparseable == 0,
        f"Negative values: {invalid}; unparseable values: {unparseable}",
    )


def validate_data(data_dir):
    """Run core data-quality checks and return a list of results."""
    data_dir = Path(data_dir)
    results = []

    # Load only the tables needed for these checks.
    customers = load_csv(data_dir, "customers")
    orders = load_csv(data_dir, "orders")
    items = load_csv(data_dir, "order_items")
    products = load_csv(data_dir, "products")
    sellers = load_csv(data_dir, "sellers")
    payments = load_csv(data_dir, "payments")
    reviews = load_csv(data_dir, "reviews")

    # 1. Primary keys and composite key.
    _check_not_null(customers, "customer_id", "Customers", results)
    _check_unique(customers, ["customer_id"], "Customers", results)

    _check_not_null(orders, "order_id", "Orders", results)
    _check_unique(orders, ["order_id"], "Orders", results)

    _check_not_null(items, "order_id", "Order items", results)
    _check_not_null(items, "order_item_id", "Order items", results)
    _check_unique(
        items,
        ["order_id", "order_item_id"],
        "Order items",
        results,
    )

    _check_not_null(products, "product_id", "Products", results)
    _check_unique(products, ["product_id"], "Products", results)

    _check_not_null(sellers, "seller_id", "Sellers", results)
    _check_unique(sellers, ["seller_id"], "Sellers", results)

    # 2. Required relationship fields.
    _check_not_null(orders, "customer_id", "Orders", results)
    _check_not_null(items, "product_id", "Order items", results)
    _check_not_null(items, "seller_id", "Order items", results)
    _check_not_null(payments, "order_id", "Payments", results)
    _check_not_null(reviews, "order_id", "Reviews", results)

    # 3. Referential integrity.
    _check_foreign_key(
        orders, "customer_id", customers, "customer_id",
        "Orders reference existing customers", results,
    )
    _check_foreign_key(
        items, "product_id", products, "product_id",
        "Order items reference existing products", results,
    )
    _check_foreign_key(
        items, "seller_id", sellers, "seller_id",
        "Order items reference existing sellers", results,
    )
    _check_foreign_key(
        items, "order_id", orders, "order_id",
        "Order items reference existing orders", results,
    )
    _check_foreign_key(
        payments, "order_id", orders, "order_id",
        "Payments reference existing orders", results,
    )
    _check_foreign_key(
        reviews, "order_id", orders, "order_id",
        "Reviews reference existing orders", results,
    )

    # 4. Numeric validity.
    for column in ["price", "freight_value"]:
        _check_non_negative(items, column, "Order items", results)

    _check_non_negative(payments, "payment_value", "Payments", results)
    _check_non_negative(
        payments, "payment_installments", "Payments", results
    )

    # 5. Review score must be an integer from 1 through 5.
    scores = pd.to_numeric(reviews["review_score"], errors="coerce")
    invalid_scores = int(
        (
            scores.isna()
            | (scores < 1)
            | (scores > 5)
            | (scores % 1 != 0)
        ).sum()
    )
    _record(
        results,
        "Reviews: scores are integers from 1 to 5",
        invalid_scores == 0,
        f"Invalid scores: {invalid_scores}",
    )

    return results


def print_report(results):
    """Print a readable validation report and summary."""
    print("=" * 72)
    print("SALES ANALYTICS PIPELINE — DATA QUALITY REPORT")
    print("=" * 72)

    for result in results:
        print(
            f"[{result['status']}] {result['check']}\n"
            f"        {result['details']}"
        )

    passed = sum(result["status"] == "PASS" for result in results)
    failed = sum(result["status"] == "FAIL" for result in results)

    print("-" * 72)
    print(f"Total checks: {len(results)}")
    print(f"Passed: {passed}")
    print(f"Failed: {failed}")
    print("=" * 72)

    return failed == 0
'''

VALIDATE_FILE.write_text(code.strip() + "\n", encoding="utf-8")

print(f"Created fresh validation module:\n{VALIDATE_FILE}")
print(f"File size: {VALIDATE_FILE.stat().st_size:,} bytes")


Created fresh validation module:
/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/validate.py
File size: 5,660 bytes


In [33]:

import importlib
import sys

REPO_ROOT = "/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline"
DATA_DIR = f"{REPO_ROOT}/data/raw/olist"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.sales_pipeline.validate as validate_module
importlib.reload(validate_module)

results = validate_module.validate_data(DATA_DIR)
all_passed = validate_module.print_report(results)

assert all_passed, (
    "Some data-quality checks failed. Review the report before proceeding."
)

print("\nSUCCESS: All data-quality checks passed.")


SALES ANALYTICS PIPELINE — DATA QUALITY REPORT
[PASS] Customers: customer_id has no nulls
        Missing values: 0
[PASS] Customers: customer_id is unique
        Duplicate rows for key: 0
[PASS] Orders: order_id has no nulls
        Missing values: 0
[PASS] Orders: order_id is unique
        Duplicate rows for key: 0
[PASS] Order items: order_id has no nulls
        Missing values: 0
[PASS] Order items: order_item_id has no nulls
        Missing values: 0
[PASS] Order items: order_id, order_item_id is unique
        Duplicate rows for key: 0
[PASS] Products: product_id has no nulls
        Missing values: 0
[PASS] Products: product_id is unique
        Duplicate rows for key: 0
[PASS] Sellers: seller_id has no nulls
        Missing values: 0
[PASS] Sellers: seller_id is unique
        Duplicate rows for key: 0
[PASS] Orders: customer_id has no nulls
        Missing values: 0
[PASS] Order items: product_id has no nulls
        Missing values: 0
[PASS] Order items: seller_id has no nul

# Data Transformation

Crete *transform.py*

In [34]:

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

TRANSFORM_FILE = REPO_ROOT / "src/sales_pipeline/transform.py"
TRANSFORM_FILE.parent.mkdir(parents=True, exist_ok=True)

code = r'''
"""Transform raw Olist data into analytics-ready datasets."""

from pathlib import Path

import pandas as pd

from .extract import load_csv


def build_order_analytics(data_dir):
    """
    Build one row per order.

    Revenue is calculated from order items; payment totals are aggregated
    independently to avoid multiplying values during joins.
    """
    data_dir = Path(data_dir)

    orders = load_csv(data_dir, "orders")
    items = load_csv(data_dir, "order_items")
    payments = load_csv(data_dir, "payments")

    # Parse timestamps; invalid or missing values remain NaT.
    date_columns = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]

    for column in date_columns:
        orders[column] = pd.to_datetime(orders[column], errors="coerce")

    # Aggregate item-level data to one row per order.
    item_metrics = (
        items.groupby("order_id", as_index=False)
        .agg(
            item_count=("order_item_id", "count"),
            unique_products=("product_id", "nunique"),
            unique_sellers=("seller_id", "nunique"),
            item_revenue=("price", "sum"),
            freight_cost=("freight_value", "sum"),
        )
    )

    # Aggregate payment-level data independently.
    payment_metrics = (
        payments.groupby("order_id", as_index=False)
        .agg(
            payment_total=("payment_value", "sum"),
            payment_records=("payment_sequential", "count"),
            payment_types=("payment_type", "nunique"),
        )
    )

    # Both aggregate tables have at most one row per order.
    analytics = orders.merge(
        item_metrics, on="order_id", how="left", validate="one_to_one"
    )
    analytics = analytics.merge(
        payment_metrics, on="order_id", how="left", validate="one_to_one"
    )

    # Orders without item or payment records remain in the output.
    count_columns = [
        "item_count",
        "unique_products",
        "unique_sellers",
        "payment_records",
        "payment_types",
    ]
    analytics[count_columns] = analytics[count_columns].fillna(0).astype("int64")

    # Keep missing financial totals as missing when no source records exist.
    # This avoids silently treating absent records as confirmed zero amounts.

    analytics["delivery_duration_days"] = (
        analytics["order_delivered_customer_date"]
        - analytics["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    analytics["estimated_delivery_duration_days"] = (
        analytics["order_estimated_delivery_date"]
        - analytics["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    analytics["delivery_delay_days"] = (
        analytics["order_delivered_customer_date"]
        - analytics["order_estimated_delivery_date"]
    ).dt.total_seconds() / 86400

    analytics["is_delivered"] = (
        analytics["order_status"].eq("delivered")
    )

    analytics["is_late_delivery"] = (
        analytics["is_delivered"]
        & analytics["delivery_delay_days"].gt(0)
    )

    return analytics


def save_order_analytics(analytics, output_dir):
    """Save order analytics as a CSV file and return its path."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "order_analytics.csv"
    analytics.to_csv(output_path, index=False)

    return output_path
'''

TRANSFORM_FILE.write_text(code.strip() + "\n", encoding="utf-8")

print(f"Created: {TRANSFORM_FILE}")
print(f"File size: {TRANSFORM_FILE.stat().st_size:,} bytes")


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/transform.py
File size: 3,494 bytes


In [35]:
#Run the transformation

import importlib
import sys
from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)
DATA_DIR = REPO_ROOT / "data/raw/olist"
PROCESSED_DIR = REPO_ROOT / "data/processed"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.transform as transform_module
importlib.reload(transform_module)

order_analytics = transform_module.build_order_analytics(DATA_DIR)

print("Transformation completed.")
print("Rows:", len(order_analytics))
print("Columns:", len(order_analytics.columns))
print("\nFirst five rows:")
display(order_analytics.head())

print("\nOrder status distribution:")
display(order_analytics["order_status"].value_counts(dropna=False))


Transformation completed.
Rows: 99441
Columns: 21

First five rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,unique_products,...,item_revenue,freight_cost,payment_total,payment_records,payment_types,delivery_duration_days,estimated_delivery_duration_days,delivery_delay_days,is_delivered,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,1,...,29.99,8.72,38.71,3,2,8.436574,15.544063,-7.107488,True,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,1,...,118.70,22.76,141.46,1,1,13.782037,19.137766,-5.355729,True,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,1,...,159.90,19.22,179.12,1,1,9.394213,26.639711,-17.245498,True,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,1,...,45.00,27.20,72.20,1,1,13.208750,26.188819,-12.980069,True,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,1,...,19.90,8.72,28.62,1,1,2.873877,12.112049,-9.238171,True,False



Order status distribution:


,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [36]:
#validate the transformed output


# Structural checks
assert len(order_analytics) == 99441, (
    f"Expected 99,441 orders, found {len(order_analytics):,}"
)

assert order_analytics["order_id"].is_unique, (
    "Duplicate order IDs found after transformation."
)

required_columns = [
    "item_revenue",
    "freight_cost",
    "payment_total",
    "delivery_duration_days",
    "delivery_delay_days",
    "is_delivered",
    "is_late_delivery",
]

missing_columns = [
    column for column in required_columns
    if column not in order_analytics.columns
]
assert not missing_columns, f"Missing columns: {missing_columns}"

# Delivered orders should have a delivery timestamp for duration calculation.
delivered = order_analytics["is_delivered"]
missing_delivered_dates = order_analytics.loc[
    delivered, "order_delivered_customer_date"
].isna().sum()

print("Unique order IDs:", order_analytics["order_id"].nunique())
print("Duplicate order IDs:", order_analytics["order_id"].duplicated().sum())
print("Delivered orders with missing delivery dates:", missing_delivered_dates)
print("Missing item revenue:", order_analytics["item_revenue"].isna().sum())
print("Missing payment totals:", order_analytics["payment_total"].isna().sum())

print("\nSUCCESS: Structural transformation checks passed.")


Unique order IDs: 99441
Duplicate order IDs: 0
Delivered orders with missing delivery dates: 8
Missing item revenue: 775
Missing payment totals: 1

SUCCESS: Structural transformation checks passed.


In [37]:
#Save the processed dataset


output_path = transform_module.save_order_analytics(
    order_analytics,
    PROCESSED_DIR,
)

print(f"Saved analytics dataset: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024 * 1024):.2f} MB")


Saved analytics dataset: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/order_analytics.csv
File size: 24.61 MB


In [41]:
#inspect the affected results

# Investigate orders with missing item revenue, payment totals,
# or delivery timestamps.

import pandas as pd

missing_revenue = order_analytics[
    order_analytics["item_revenue"].isna()
]

missing_payment = order_analytics[
    order_analytics["payment_total"].isna()
]

missing_delivery = order_analytics[
    order_analytics["is_delivered"]
    & order_analytics["order_delivered_customer_date"].isna()
]

print("ORDERS WITH MISSING ITEM REVENUE")
display(
    missing_revenue[
        ["order_id", "customer_id", "order_status",
         "item_count", "payment_total"]
    ]
)

print("\nORDERS WITH MISSING PAYMENT TOTAL")
display(
    missing_payment[
        ["order_id", "customer_id", "order_status",
         "item_count", "item_revenue"]
    ]
)

print("\nDELIVERED ORDERS WITH MISSING DELIVERY DATES")
display(
    missing_delivery[
        ["order_id", "customer_id", "order_status",
         "order_purchase_timestamp",
         "order_delivered_customer_date"]
    ]
)

print("\nMissing revenue by order status")
display(missing_revenue["order_status"].value_counts(dropna=False))

print("\nMissing payment by order status")
display(missing_payment["order_status"].value_counts(dropna=False))


from src.sales_pipeline.extract import load_csv

orders_source = load_csv(DATA_DIR, "orders")
payments_source = load_csv(DATA_DIR, "payments")
items_source = load_csv(DATA_DIR, "order_items")

# Eight delivered orders with missing customer delivery dates
delivery_ids = missing_delivery["order_id"]

delivery_source_check = orders_source[
    orders_source["order_id"].isin(delivery_ids)
][[
    "order_id",
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]]

print("Source timestamps for affected delivered orders:")
display(delivery_source_check)

# One delivered order without a payment record
payment_ids = missing_payment["order_id"]

print("\nSource payment records for the affected order:")
display(
    payments_source[
        payments_source["order_id"].isin(payment_ids)
    ]
)

print("\nSource order-item records for the affected order:")
display(
    items_source[
        items_source["order_id"].isin(payment_ids)
    ]
)


# Add explicit data-quality indicators without changing source values.

order_analytics["has_item_records"] = order_analytics["item_count"] > 0

order_analytics["has_payment_records"] = (
    order_analytics["payment_records"] > 0
)

order_analytics["missing_delivery_timestamp"] = (
    order_analytics["is_delivered"]
    & order_analytics["order_delivered_customer_date"].isna()
)

order_analytics["missing_item_revenue"] = (
    order_analytics["item_revenue"].isna()
)

order_analytics["missing_payment_total"] = (
    order_analytics["payment_total"].isna()
)

# Summarize the exceptions for documentation.
quality_summary = pd.DataFrame({
    "metric": [
        "Total orders",
        "Orders without item records",
        "Orders without payment records",
        "Delivered orders missing delivery timestamps",
        "Orders with missing item revenue",
        "Orders with missing payment totals",
    ],
    "count": [
        len(order_analytics),
        (~order_analytics["has_item_records"]).sum(),
        (~order_analytics["has_payment_records"]).sum(),
        order_analytics["missing_delivery_timestamp"].sum(),
        order_analytics["missing_item_revenue"].sum(),
        order_analytics["missing_payment_total"].sum(),
    ],
})

display(quality_summary)

output_path = transform_module.save_order_analytics(
    order_analytics, PROCESSED_DIR
)

print(f"\nUpdated processed dataset: {output_path}")



ORDERS WITH MISSING ITEM REVENUE


,order_id,customer_id,order_status,item_count,payment_total
266,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,0,84.00
586,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,0,97.68
687,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,0,132.46
737,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,0,134.38
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,0,0.00
...,...,...,...,...,...
99252,aaab15da689073f8f9aa978a390a69d1,df20748206e4b865b2f14a5eabbfcf34,unavailable,0,210.09
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,0,63.89
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,0,107.13
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,0,195.91



ORDERS WITH MISSING PAYMENT TOTAL


,order_id,customer_id,order_status,item_count,item_revenue
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,3,134.97



DELIVERED ORDERS WITH MISSING DELIVERY DATES


...

,order_id,payment_sequential,payment_type,payment_installments,payment_value



Source order-item records for the affected order:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83


,metric,count
0,Total orders,99441
1,Orders without item records,775
2,Orders without payment records,1
3,Delivered orders missing delivery timestamps,8
4,Orders with missing item revenue,775
5,Orders with missing payment totals,1



Updated processed dataset: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/order_analytics.csv


# Product and Category Analytics

Create *transform_products.py*


In [42]:

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

PRODUCT_TRANSFORM_FILE = (
    REPO_ROOT / "src/sales_pipeline/transform_products.py"
)

code = r'''
"""Product and category analytics transformations."""

from pathlib import Path

import pandas as pd

from .extract import load_csv


EXCLUDED_SALES_STATUSES = {"canceled", "unavailable"}


def build_product_analytics(data_dir):
    """Build product- and category-level sales analytics."""
    data_dir = Path(data_dir)

    items = load_csv(data_dir, "order_items")
    products = load_csv(data_dir, "products")
    translations = load_csv(data_dir, "category_translation")
    orders = load_csv(data_dir, "orders")

    # Join order status to each item before calculating sales metrics.
    sales_items = items.merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left",
        validate="many_to_one",
    )

    # Exclude cancelled and unavailable orders from sales metrics.
    sales_items = sales_items[
        ~sales_items["order_status"].isin(EXCLUDED_SALES_STATUSES)
    ].copy()

    # Add product attributes and translated category names.
    sales_items = sales_items.merge(
        products,
        on="product_id",
        how="left",
        validate="many_to_one",
    )

    sales_items = sales_items.merge(
        translations,
        on="product_category_name",
        how="left",
        validate="many_to_one",
    )

    # Keep unmapped categories visible instead of dropping them.
    sales_items["product_category_name_english"] = (
        sales_items["product_category_name_english"]
        .fillna("unknown_or_untranslated")
    )

    product_analytics = (
        sales_items.groupby(
            ["product_id", "product_category_name_english"],
            as_index=False,
            dropna=False,
        )
        .agg(
            units_sold=("order_item_id", "count"),
            item_revenue=("price", "sum"),
            freight_cost=("freight_value", "sum"),
            distinct_orders=("order_id", "nunique"),
            distinct_sellers=("seller_id", "nunique"),
        )
        .sort_values("item_revenue", ascending=False)
        .reset_index(drop=True)
    )

    category_analytics = (
        sales_items.groupby(
            "product_category_name_english",
            as_index=False,
            dropna=False,
        )
        .agg(
            units_sold=("order_item_id", "count"),
            item_revenue=("price", "sum"),
            freight_cost=("freight_value", "sum"),
            distinct_orders=("order_id", "nunique"),
            distinct_products=("product_id", "nunique"),
            distinct_sellers=("seller_id", "nunique"),
        )
        .sort_values("item_revenue", ascending=False)
        .reset_index(drop=True)
    )

    return product_analytics, category_analytics


def save_product_analytics(product_analytics, category_analytics, output_dir):
    """Save product- and category-level analytics CSV files."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    product_path = output_dir / "product_analytics.csv"
    category_path = output_dir / "category_analytics.csv"

    product_analytics.to_csv(product_path, index=False)
    category_analytics.to_csv(category_path, index=False)

    return product_path, category_path
'''

PRODUCT_TRANSFORM_FILE.write_text(
    code.strip() + "\n",
    encoding="utf-8",
)

print(f"Created: {PRODUCT_TRANSFORM_FILE}")
print(f"File size: {PRODUCT_TRANSFORM_FILE.stat().st_size:,} bytes")


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/transform_products.py
File size: 3,191 bytes


In [43]:
#Run the transformation

import importlib
import sys

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)
DATA_DIR = REPO_ROOT / "data/raw/olist"
PROCESSED_DIR = REPO_ROOT / "data/processed"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.transform_products as product_transform
importlib.reload(product_transform)

product_analytics, category_analytics = (
    product_transform.build_product_analytics(DATA_DIR)
)

print("Product analytics shape:", product_analytics.shape)
print("Category analytics shape:", category_analytics.shape)

print("\nTop 10 products by item revenue:")
display(product_analytics.head(10))

print("\nTop 10 categories by item revenue:")
display(category_analytics.head(10))


Product analytics shape: (32729, 7)
Category analytics shape: (72, 7)

Top 10 products by item revenue:


,product_id,product_category_name_english,units_sold,item_revenue,freight_cost,distinct_orders,distinct_sellers
0,bb50f2e236e5eea0100680137654686c,health_beauty,195,63885.00,3721.10,187,1
1,6cdd53843498f92890544667809f1595,health_beauty,156,54730.20,4363.79,151,1
2,d6160fb7873f184099d9bc95e30376af,computers,35,48899.34,1426.84,35,1
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,341,46916.51,13680.78,321,1
4,99a4788cb24856965c36a24e339b6058,bed_bath_table,487,42938.66,8024.94,466,2
5,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,274,41082.60,7129.62,255,1
6,25c38557cf793876c5abdd5931f922db,baby,38,38907.32,1404.63,38,2
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,37733.90,3991.91,63,1
8,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,323,37683.42,2274.51,306,3
9,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527,37608.90,7211.86,431,1



Top 10 categories by item revenue:


,product_category_name_english,units_sold,item_revenue,freight_cost,distinct_orders,distinct_products,distinct_sellers
0,health_beauty,9634,1255695.13,181970.65,8800,2430,489
1,watches_gifts,5970,1198185.21,100107.26,5604,1322,98
2,bed_bath_table,11097,1035964.06,204422.07,9399,3022,195
3,sports_leisure,8590,979740.92,167503.71,7673,2850,477
4,computers_accessories,7781,904322.02,146619.56,6654,1630,285
5,furniture_decor,8298,727465.05,172160.99,6425,2641,366
6,housewares,6915,626825.80,145209.34,5847,2314,463
7,cool_stuff,3779,620770.49,83315.75,3616,783,263
8,auto,4204,586585.73,92020.91,3872,1882,374
9,garden_tools,4328,481009.94,98515.26,3505,748,235


In [44]:
#Validate and save the outputs

# Basic structural and aggregation checks.
assert product_analytics["product_id"].notna().all()
assert product_analytics["product_id"].is_unique

assert category_analytics[
    "product_category_name_english"
].is_unique

assert (product_analytics["units_sold"] >= 0).all()
assert (category_analytics["units_sold"] >= 0).all()

# Product and category aggregations should reconcile.
assert (
    product_analytics["units_sold"].sum()
    == category_analytics["units_sold"].sum()
)

assert abs(
    product_analytics["item_revenue"].sum()
    - category_analytics["item_revenue"].sum()
) < 0.01

product_path, category_path = (
    product_transform.save_product_analytics(
        product_analytics,
        category_analytics,
        PROCESSED_DIR,
    )
)

print("SUCCESS: Product/category validation passed.")
print("Product file:", product_path)
print("Category file:", category_path)
print(f"Product file size: {product_path.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Category file size: {category_path.stat().st_size / (1024 * 1024):.2f} MB")


SUCCESS: Product/category validation passed.
Product file: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/product_analytics.csv
Category file: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/category_analytics.csv
Product file size: 2.08 MB
Category file size: 0.00 MB


# Customer Analytics

Create *transformation files*

In [49]:

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

CUSTOMER_TRANSFORM_FILE = (
    REPO_ROOT / "src/sales_pipeline/transform_customers.py"
)

code = r'''
"""Customer-level analytics transformations."""

from pathlib import Path
import pandas as pd
from .extract import load_csv

EXCLUDED_SALES_STATUSES = {"canceled", "unavailable"}



def build_customer_analytics(data_dir):
    data_dir = Path(data_dir)

    customers = load_csv(data_dir, "customers")
    orders = load_csv(data_dir, "orders")
    items = load_csv(data_dir, "order_items")

    eligible_orders = orders[
        ~orders["order_status"].isin(EXCLUDED_SALES_STATUSES)
    ].copy()

    eligible_orders["order_purchase_timestamp"] = pd.to_datetime(
        eligible_orders["order_purchase_timestamp"],
        errors="coerce"
    )

    # Calculate item revenue at order level before joining.
    order_revenue = (
        items.groupby("order_id", as_index=False)
        .agg(item_revenue=("price", "sum"))
    )

    eligible_orders = eligible_orders.merge(
        order_revenue,
        on="order_id",
        how="left",
        validate="one_to_one"
    )

    # Attach customer identity and state to each order.
    customer_orders = eligible_orders.merge(
        customers[
            ["customer_id", "customer_unique_id", "customer_state"]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )

    # Aggregate metrics across all states for each customer.
    metrics = (
        customer_orders.groupby(
            "customer_unique_id",
            as_index=False,
            dropna=False
        )
        .agg(
            order_count=("order_id", "nunique"),
            item_revenue=("item_revenue", "sum"),
            first_order_date=("order_purchase_timestamp", "min"),
            last_order_date=("order_purchase_timestamp", "max"),
            orders_with_revenue=("item_revenue", "count"),
        )
    )

    # Obtain the state from the most recent order.
    latest_orders = (
        customer_orders
        .sort_values(
            ["order_purchase_timestamp", "order_id"],
            na_position="first"
        )
        .drop_duplicates("customer_unique_id", keep="last")
        [["customer_unique_id", "customer_state"]]
    )

    customer_analytics = metrics.merge(
        latest_orders,
        on="customer_unique_id",
        how="left",
        validate="one_to_one"
    )

    customer_analytics["average_order_value"] = (
        customer_analytics["item_revenue"]
        / customer_analytics["orders_with_revenue"].replace(0, pd.NA)
    )

    customer_analytics["is_repeat_customer"] = (
        customer_analytics["order_count"] > 1
    )

    customer_analytics = customer_analytics.sort_values(
        "item_revenue",
        ascending=False
    ).reset_index(drop=True)

    return customer_analytics



def save_customer_analytics(customer_analytics, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "customer_analytics.csv"
    customer_analytics.to_csv(output_path, index=False)

    return output_path
'''

CUSTOMER_TRANSFORM_FILE.write_text(
    code.strip() + "\n",
    encoding="utf-8"
)

print("Created:", CUSTOMER_TRANSFORM_FILE)


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/transform_customers.py


In [51]:
# Check whether a customer_unique_id maps to multiple states.
import importlib
import src.sales_pipeline.transform_customers as customer_transform

importlib.reload(customer_transform)

customer_analytics = customer_transform.build_customer_analytics(DATA_DIR)

assert customer_analytics["customer_unique_id"].notna().all()
assert customer_analytics["customer_unique_id"].is_unique
assert (customer_analytics["order_count"] >= 1).all()

assert (
    customer_analytics["is_repeat_customer"]
    == (customer_analytics["order_count"] > 1)
).all()

assert (
    customer_analytics["orders_with_revenue"]
    <= customer_analytics["order_count"]
).all()

customer_path = customer_transform.save_customer_analytics(
    customer_analytics, PROCESSED_DIR
)

print("Customer rows:", len(customer_analytics))
print(
    "Distinct customer IDs:",
    customer_analytics["customer_unique_id"].nunique()
)
print("Duplicate customer IDs:", customer_analytics["customer_unique_id"].duplicated().sum())
print("Validation: PASSED")
print("Saved:", customer_path)

display(customer_analytics.head(10))


Customer rows: 94990
Distinct customer IDs: 94990
Duplicate customer IDs: 0
Validation: PASSED
Saved: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/customer_analytics.csv


,customer_unique_id,order_count,item_revenue,first_order_date,last_order_date,orders_with_revenue,customer_state,average_order_value,is_repeat_customer
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.0,2017-09-29 15:24:52,2017-09-29 15:24:52,1,RJ,13440.0,False
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.0,2017-04-01 15:58:40,2017-04-01 15:58:41,2,RJ,3694.0,True
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.0,2018-07-15 14:49:44,2018-07-15 14:49:44,1,ES,7160.0,False
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.0,2017-02-12 20:37:36,2017-02-12 20:37:36,1,MS,6735.0,False
4,459bef486812aa25204be022145caa62,1,6729.0,2018-07-25 18:10:17,2018-07-25 18:10:17,1,ES,6729.0,False
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.0,2017-05-24 18:14:34,2017-05-24 18:14:34,1,SP,6499.0,False
6,4007669dec559734d6f53e029e360987,1,5934.6,2017-11-24 11:03:35,2017-11-24 11:03:35,1,MG,5934.6,False
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.0,2017-04-18 18:50:13,2017-04-18 18:50:13,1,SP,4690.0,False
8,48e1ac109decbb87765a3eade6854098,1,4590.0,2018-06-22 12:23:19,2018-06-22 12:23:19,1,PB,4590.0,False
9,a229eba70ec1c2abef51f04987deb7a5,1,4400.0,2018-05-31 22:57:07,2018-05-31 22:57:07,1,RJ,4400.0,False


In [54]:
# run and inspect the transformation

import importlib
import sys

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = REPO_ROOT / "data/raw/olist"
PROCESSED_DIR = REPO_ROOT / "data/processed"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.transform_customers as customer_transform

importlib.reload(customer_transform)

customer_analytics = (
    customer_transform.build_customer_analytics(DATA_DIR)
)

print("Customer analytics shape:", customer_analytics.shape)
print(
    "Unique customers:",
    customer_analytics["customer_unique_id"].nunique()
)

display(customer_analytics.head(10))


Customer analytics shape: (94990, 9)
Unique customers: 94990


,customer_unique_id,order_count,item_revenue,first_order_date,last_order_date,orders_with_revenue,customer_state,average_order_value,is_repeat_customer
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.0,2017-09-29 15:24:52,2017-09-29 15:24:52,1,RJ,13440.0,False
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.0,2017-04-01 15:58:40,2017-04-01 15:58:41,2,RJ,3694.0,True
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.0,2018-07-15 14:49:44,2018-07-15 14:49:44,1,ES,7160.0,False
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.0,2017-02-12 20:37:36,2017-02-12 20:37:36,1,MS,6735.0,False
4,459bef486812aa25204be022145caa62,1,6729.0,2018-07-25 18:10:17,2018-07-25 18:10:17,1,ES,6729.0,False
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.0,2017-05-24 18:14:34,2017-05-24 18:14:34,1,SP,6499.0,False
6,4007669dec559734d6f53e029e360987,1,5934.6,2017-11-24 11:03:35,2017-11-24 11:03:35,1,MG,5934.6,False
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.0,2017-04-18 18:50:13,2017-04-18 18:50:13,1,SP,4690.0,False
8,48e1ac109decbb87765a3eade6854098,1,4590.0,2018-06-22 12:23:19,2018-06-22 12:23:19,1,PB,4590.0,False
9,a229eba70ec1c2abef51f04987deb7a5,1,4400.0,2018-05-31 22:57:07,2018-05-31 22:57:07,1,RJ,4400.0,False


In [55]:
#validate and save the output

assert customer_analytics["customer_unique_id"].notna().all()
assert not customer_analytics.duplicated(
    ["customer_unique_id", "customer_state"]
).any()

assert (customer_analytics["order_count"] >= 1).all()
assert (
    customer_analytics["is_repeat_customer"]
    == (customer_analytics["order_count"] > 1)
).all()

assert (
    customer_analytics["orders_with_revenue"]
    <= customer_analytics["order_count"]
).all()

customer_path = customer_transform.save_customer_analytics(
    customer_analytics,
    PROCESSED_DIR
)

print("SUCCESS: Customer analytics validation passed.")
print("Output file:", customer_path)
print("File size (MB):", round(customer_path.stat().st_size / 1_000_000, 2))



SUCCESS: Customer analytics validation passed.
Output file: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/customer_analytics.csv
File size (MB): 9.28


# Sales Performance Analysis

create *tranform_sales.py*

In [56]:

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SALES_TRANSFORM_FILE = REPO_ROOT / "src/sales_pipeline/transform_sales.py"

code = r'''
"""Sales performance analytics transformations."""

from pathlib import Path
import pandas as pd
from .extract import load_csv

EXCLUDED_SALES_STATUSES = {"canceled", "unavailable"}


def build_sales_analytics(data_dir):
    data_dir = Path(data_dir)

    orders = load_csv(data_dir, "orders")
    items = load_csv(data_dir, "order_items")

    orders["order_purchase_timestamp"] = pd.to_datetime(
        orders["order_purchase_timestamp"],
        errors="coerce"
    )

    # Aggregate item revenue once per order to avoid join multiplication.
    order_revenue = (
        items.groupby("order_id", as_index=False)
        .agg(
            item_revenue=("price", "sum"),
            freight_cost=("freight_value", "sum"),
            item_count=("order_item_id", "count"),
        )
    )

    orders = orders.merge(
        order_revenue,
        on="order_id",
        how="left",
        validate="one_to_one"
    )

    # Monthly KPIs use eligible orders only.
    eligible_orders = orders[
        ~orders["order_status"].isin(EXCLUDED_SALES_STATUSES)
    ].copy()

    eligible_orders["order_month"] = (
        eligible_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
    )

    monthly_sales = (
        eligible_orders.dropna(subset=["order_purchase_timestamp"])
        .groupby("order_month", as_index=False)
        .agg(
            order_count=("order_id", "nunique"),
            item_revenue=("item_revenue", "sum"),
            freight_cost=("freight_cost", "sum"),
            total_items=("item_count", "sum"),
            orders_with_revenue=("item_revenue", "count"),
        )
    )

    monthly_sales["average_order_value"] = (
        monthly_sales["item_revenue"]
        / monthly_sales["orders_with_revenue"].replace(0, pd.NA)
    )

    monthly_sales = monthly_sales.sort_values("order_month").reset_index(drop=True)

    # Keep the complete status distribution separately for operational analysis.
    status_summary = (
        orders.groupby("order_status", as_index=False, dropna=False)
        .agg(order_count=("order_id", "nunique"))
        .sort_values("order_count", ascending=False)
        .reset_index(drop=True)
    )

    return monthly_sales, status_summary


def save_sales_analytics(monthly_sales, status_summary, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    monthly_path = output_dir / "monthly_sales_analytics.csv"
    status_path = output_dir / "order_status_summary.csv"

    monthly_sales.to_csv(monthly_path, index=False)
    status_summary.to_csv(status_path, index=False)

    return monthly_path, status_path
'''

SALES_TRANSFORM_FILE.write_text(code.strip() + "\n", encoding="utf-8")

print("Created:", SALES_TRANSFORM_FILE)


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/transform_sales.py


In [57]:
#Run the transformation

import importlib
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.transform_sales as sales_transform
importlib.reload(sales_transform)

monthly_sales, status_summary = sales_transform.build_sales_analytics(DATA_DIR)

print("Monthly sales shape:", monthly_sales.shape)
print("Order status summary shape:", status_summary.shape)

display(monthly_sales.head(10))
display(status_summary)


Monthly sales shape: (24, 7)
Order status summary shape: (8, 2)


,order_month,order_count,item_revenue,freight_cost,total_items,orders_with_revenue,average_order_value
0,2016-09,2,207.86,71.83,5.0,2,103.930000
1,2016-10,293,44507.30,6847.22,342.0,290,153.473448
2,2016-12,1,10.90,8.72,1.0,1,10.900000
3,2017-01,787,120098.27,16845.19,953.0,787,152.602630
4,2017-02,1718,244959.35,38602.34,1936.0,1718,142.584022
5,2017-03,2617,368341.32,57276.64,2975.0,2617,140.749454
6,2017-04,2377,353842.98,52005.63,2660.0,2377,148.861161
7,2017-05,3640,503159.19,79551.64,4106.0,3640,138.230547
8,2017-06,3205,429916.61,69735.63,3571.0,3205,134.139348
9,2017-07,3946,492287.30,86466.43,4493.0,3946,124.756031


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [59]:
#validate and save

assert monthly_sales["order_month"].is_unique
assert monthly_sales["order_month"].is_monotonic_increasing
assert (monthly_sales["order_count"] >= 0).all()
assert (monthly_sales["total_items"].fillna(0) >= 0).all()

assert status_summary["order_status"].is_unique
assert status_summary["order_count"].sum() == 99441

monthly_path, status_path = sales_transform.save_sales_analytics(
    monthly_sales,
    status_summary,
    PROCESSED_DIR
)

print("SUCCESS: Sales analytics validation passed.")
print("Monthly sales:", monthly_path)
print("Order status summary:", status_path)


SUCCESS: Sales analytics validation passed.
Monthly sales: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/monthly_sales_analytics.csv
Order status summary: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/order_status_summary.csv


In [60]:

# Check revenue completeness in the order-level dataset.
# This uses the same eligible-order definition as the sales module.

import pandas as pd

orders_raw = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    low_memory=False
)
items_raw = pd.read_csv(
    DATA_DIR / "olist_order_items_dataset.csv",
    low_memory=False
)

order_revenue_check = (
    items_raw.groupby("order_id", as_index=False)
    .agg(item_revenue=("price", "sum"))
)

orders_check = orders_raw.merge(
    order_revenue_check,
    on="order_id",
    how="left",
    validate="one_to_one"
)

eligible_check = orders_check[
    ~orders_check["order_status"].isin(
        {"canceled", "unavailable"}
    )
]

print("Eligible orders:", len(eligible_check))
print("Eligible orders without item records:",
      int(eligible_check["item_revenue"].isna().sum()))
print("Eligible orders with item records:",
      int(eligible_check["item_revenue"].notna().sum()))
print("Eligible orders with zero item revenue:",
      int(eligible_check["item_revenue"].eq(0).sum()))


Eligible orders: 98207
Eligible orders without item records: 8
Eligible orders with item records: 98199
Eligible orders with zero item revenue: 0


In [61]:
from pathlib import Path

PROCESSED_DIR = REPO_ROOT / "data/processed"

expected_outputs = [
    "order_analytics.csv",
    "product_analytics.csv",
    "category_analytics.csv",
    "customer_analytics.csv",
    "monthly_sales_analytics.csv",
    "order_status_summary.csv",
]

for filename in expected_outputs:
    path = PROCESSED_DIR / filename
    print(f"{'OK' if path.exists() else 'MISSING'}: {filename}")

OK: order_analytics.csv
OK: product_analytics.csv
OK: category_analytics.csv
OK: customer_analytics.csv
OK: monthly_sales_analytics.csv
OK: order_status_summary.csv


# Pipeline Orchestration

Create pipeline.py

In [66]:

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

PIPELINE_FILE = REPO_ROOT / "src/sales_pipeline/pipeline.py"

code = r'''
"""Orchestrate the sales analytics data pipeline."""

from pathlib import Path
from datetime import datetime, timezone

from . import validate
from . import transform
from . import transform_products
from . import transform_customers
from . import transform_sales


def run_pipeline(data_dir, output_dir):
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    started_at = datetime.now(timezone.utc)

    print("=" * 64)
    print("SALES ANALYTICS DATA PIPELINE")
    print("=" * 64)

    # Step 1: Validate source data.
    print("\n[1/5] Validating source data...")
    validation_results = validate.validate_data(data_dir)
    validation_passed = validate.print_report(validation_results)

    if not validation_passed:
        raise RuntimeError(
            "Pipeline stopped: one or more data-quality checks failed."
        )

    # Step 2: Order analytics.
    print("\n[2/5] Building order analytics...")
    order_analytics = transform.build_order_analytics(data_dir)
    order_path = transform.save_order_analytics(
        order_analytics, output_dir
    )

    # Step 3: Product and category analytics.
    print("\n[3/5] Building product and category analytics...")
    product_analytics, category_analytics = (
        transform_products.build_product_analytics(data_dir)
    )
    product_path, category_path = (
        transform_products.save_product_analytics(
            product_analytics, category_analytics, output_dir
        )
    )

    # Step 4: Customer analytics.
    print("\n[4/5] Building customer analytics...")
    customer_analytics = (
        transform_customers.build_customer_analytics(data_dir)
    )
    customer_path = transform_customers.save_customer_analytics(
        customer_analytics, output_dir
    )

    # Step 5: Monthly sales and order-status summary.
    print("\n[5/5] Building sales performance analytics...")
    monthly_sales, status_summary = (
        transform_sales.build_sales_analytics(data_dir)
    )
    monthly_path, status_path = (
        transform_sales.save_sales_analytics(
            monthly_sales, status_summary, output_dir
        )
    )

    outputs = [
        order_path,
        product_path,
        category_path,
        customer_path,
        monthly_path,
        status_path,
    ]

    # Verify that every expected output exists and is non-empty.
    for path in outputs:
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(
                f"Pipeline output is missing or empty: {path}"
            )

    finished_at = datetime.now(timezone.utc)
    duration = (finished_at - started_at).total_seconds()

    summary = {
        "started_at": started_at.isoformat(),
        "finished_at": finished_at.isoformat(),
        "duration_seconds": round(duration, 2),
        "validation_checks": len(validation_results),
        "order_rows": len(order_analytics),
        "product_rows": len(product_analytics),
        "category_rows": len(category_analytics),
        "customer_rows": len(customer_analytics),
        "monthly_rows": len(monthly_sales),
        "status_rows": len(status_summary),
        "outputs": [str(path) for path in outputs],
    }

    print("\n" + "=" * 64)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 64)
    print(f"Duration: {summary['duration_seconds']} seconds")
    print(f"Validation checks: {summary['validation_checks']}")
    print(f"Order rows: {summary['order_rows']}")
    print(f"Product rows: {summary['product_rows']}")
    print(f"Category rows: {summary['category_rows']}")
    print(f"Customer rows: {summary['customer_rows']}")
    print(f"Monthly records: {summary['monthly_rows']}")
    print(f"Order status records: {summary['status_rows']}")

    print("\nGenerated outputs:")
    for path in outputs:
        print(f"- {path} ({path.stat().st_size:,} bytes)")

    return summary
'''

PIPELINE_FILE.write_text(code.strip() + "\n", encoding="utf-8")
print("Updated:", PIPELINE_FILE)


Updated: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/pipeline.py


In [67]:
# Execute the complete pipeline

import importlib
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.sales_pipeline.pipeline as pipeline_module
importlib.reload(pipeline_module)

pipeline_summary = pipeline_module.run_pipeline(
    data_dir=REPO_ROOT / "data/raw/olist",
    output_dir=REPO_ROOT / "data/processed",
)

print("\nReturned summary:")
pipeline_summary


SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...
SALES ANALYTICS PIPELINE — DATA QUALITY REPORT
[PASS] Customers: customer_id has no nulls
        Missing values: 0
[PASS] Customers: customer_id is unique
        Duplicate rows for key: 0
[PASS] Orders: order_id has no nulls
        Missing values: 0
[PASS] Orders: order_id is unique
        Duplicate rows for key: 0
[PASS] Order items: order_id has no nulls
        Missing values: 0
[PASS] Order items: order_item_id has no nulls
        Missing values: 0
[PASS] Order items: order_id, order_item_id is unique
        Duplicate rows for key: 0
[PASS] Products: product_id has no nulls
        Missing values: 0
[PASS] Products: product_id is unique
        Duplicate rows for key: 0
[PASS] Sellers: seller_id has no nulls
        Missing values: 0
[PASS] Sellers: seller_id is unique
        Duplicate rows for key: 0
[PASS] Orders: customer_id has no nulls
        Missing values: 0
[PASS] Order items: product_id has no nulls
   

{'started_at': '2026-10-15T10:32:07.579525+00:00',
 'finished_at': '2026-10-15T10:32:24.799648+00:00',
 'duration_seconds': 17.22,
 'validation_checks': 27,
 'order_rows': 99441,
 'product_rows': 32729,
 'category_rows': 72,
 'customer_rows': 94990,
 'monthly_rows': 24,
 'status_rows': 8,
 'outputs': ['/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/order_analytics.csv',
  '/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/product_analytics.csv',
  '/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/category_analytics.csv',
  '/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/customer_analytics.csv',
  '/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/processed/monthly_sales_analytics.csv',
  '/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analyt

# Automated testing


Create Test directory

In [71]:

from pathlib import Path

TEST_DIR = REPO_ROOT / "tests"
TEST_DIR.mkdir(parents=True, exist_ok=True)

print("Test directory ready:", TEST_DIR)


Test directory ready: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests


Create test_pipeline.py

In [72]:

test_code = r'''
"""Tests for sales analytics pipeline outputs."""

import unittest
from pathlib import Path
import pandas as pd


REPO_ROOT = Path(__file__).resolve().parents[1]
PROCESSED_DIR = REPO_ROOT / "data" / "processed"


class TestPipelineOutputs(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.orders = pd.read_csv(
            PROCESSED_DIR / "order_analytics.csv"
        )
        cls.products = pd.read_csv(
            PROCESSED_DIR / "product_analytics.csv"
        )
        cls.categories = pd.read_csv(
            PROCESSED_DIR / "category_analytics.csv"
        )
        cls.customers = pd.read_csv(
            PROCESSED_DIR / "customer_analytics.csv"
        )
        cls.monthly = pd.read_csv(
            PROCESSED_DIR / "monthly_sales_analytics.csv"
        )
        cls.status = pd.read_csv(
            PROCESSED_DIR / "order_status_summary.csv"
        )

    def test_order_ids_are_unique(self):
        self.assertEqual(
            self.orders["order_id"].nunique(),
            len(self.orders)
        )

    def test_product_ids_are_unique(self):
        self.assertTrue(self.products["product_id"].is_unique)

    def test_category_names_are_unique(self):
        self.assertTrue(
            self.categories[
                "product_category_name_english"
            ].is_unique
        )

    def test_customer_ids_are_unique(self):
        self.assertTrue(
            self.customers["customer_unique_id"].is_unique
        )

    def test_customer_repeat_flag(self):
        expected = self.customers["order_count"] > 1
        actual = self.customers["is_repeat_customer"].astype(bool)
        pd.testing.assert_series_equal(
            actual.reset_index(drop=True),
            expected.reset_index(drop=True),
            check_names=False
        )

    def test_product_and_category_units_reconcile(self):
        self.assertEqual(
            self.products["units_sold"].sum(),
            self.categories["units_sold"].sum()
        )

    def test_product_and_category_revenue_reconcile(self):
        self.assertAlmostEqual(
            self.products["item_revenue"].sum(),
            self.categories["item_revenue"].sum(),
            places=2
        )

    def test_order_status_counts_reconcile(self):
        self.assertEqual(
            int(self.status["order_count"].sum()),
            99441
        )

    def test_monthly_order_month_is_unique(self):
        self.assertTrue(self.monthly["order_month"].is_unique)

    def test_expected_outputs_exist(self):
        expected_files = [
            "order_analytics.csv",
            "product_analytics.csv",
            "category_analytics.csv",
            "customer_analytics.csv",
            "monthly_sales_analytics.csv",
            "order_status_summary.csv",
        ]
        for filename in expected_files:
            with self.subTest(filename=filename):
                path = PROCESSED_DIR / filename
                self.assertTrue(path.is_file())
                self.assertGreater(path.stat().st_size, 0)


if __name__ == "__main__":
    unittest.main()
'''

TEST_FILE = TEST_DIR / "test_pipeline.py"
TEST_FILE.write_text(test_code.strip() + "\n", encoding="utf-8")

print("Created:", TEST_FILE)


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_pipeline.py


In [73]:
#Run the test

import subprocess

result = subprocess.run(
    ["python", "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)



test_category_names_are_unique (test_pipeline.TestPipelineOutputs.test_category_names_are_unique) ... ok
test_customer_ids_are_unique (test_pipeline.TestPipelineOutputs.test_customer_ids_are_unique) ... ok
test_customer_repeat_flag (test_pipeline.TestPipelineOutputs.test_customer_repeat_flag) ... ok
test_expected_outputs_exist (test_pipeline.TestPipelineOutputs.test_expected_outputs_exist) ... ok
test_monthly_order_month_is_unique (test_pipeline.TestPipelineOutputs.test_monthly_order_month_is_unique) ... ok
test_order_ids_are_unique (test_pipeline.TestPipelineOutputs.test_order_ids_are_unique) ... ok
test_order_status_counts_reconcile (test_pipeline.TestPipelineOutputs.test_order_status_counts_reconcile) ... ok
test_product_and_category_revenue_reconcile (test_pipeline.TestPipelineOutputs.test_product_and_category_revenue_reconcile) ... ok
test_product_and_category_units_reconcile (test_pipeline.TestPipelineOutputs.test_product_and_category_units_reconcile) ... ok
test_product_ids_are

# Varify and Git commit

In [75]:
# Stage the test file and review Git

import subprocess

def git_command(args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()

# Stage the new automated test.
git_command(["add", "--", "tests/test_pipeline.py"])

print("=== Staged changes ===")
print(git_command(["diff", "--cached", "--stat"]))

print("\n=== Git status ===")
print(git_command(["status", "--short"]))


=== Staged changes ===
01_data_ingestion.ipynb                   |   1 +
 src/sales_pipeline/extract.py             | 115 ++++++++++++++++++
 src/sales_pipeline/pipeline.py            | 121 +++++++++++++++++++
 src/sales_pipeline/transform.py           | 113 ++++++++++++++++++
 src/sales_pipeline/transform_customers.py | 109 +++++++++++++++++
 src/sales_pipeline/transform_products.py  | 105 +++++++++++++++++
 src/sales_pipeline/transform_sales.py     |  87 ++++++++++++++
 src/sales_pipeline/validate.py            | 186 ++++++++++++++++++++++++++++++
 tests/test_pipeline.py                    | 104 +++++++++++++++++
 9 files changed, 941 insertions(+)

=== Git status ===
AM 01_data_ingestion.ipynb
A  src/sales_pipeline/extract.py
A  src/sales_pipeline/pipeline.py
A  src/sales_pipeline/transform.py
A  src/sales_pipeline/transform_customers.py
A  src/sales_pipeline/transform_products.py
A  src/sales_pipeline/transform_sales.py
A  src/sales_pipeline/validate.py
A  tests/test_pipeline.py


In [76]:
# Restage the notebook and varify
import subprocess

def git_command(args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()

git_command(["add", "--", "01_data_ingestion.ipynb"])

print(git_command(["status", "--short"]))
print("\n=== Final staged summary ===")
print(git_command(["diff", "--cached", "--stat"]))


A  01_data_ingestion.ipynb
A  src/sales_pipeline/extract.py
A  src/sales_pipeline/pipeline.py
A  src/sales_pipeline/transform.py
A  src/sales_pipeline/transform_customers.py
A  src/sales_pipeline/transform_products.py
A  src/sales_pipeline/transform_sales.py
A  src/sales_pipeline/validate.py
A  tests/test_pipeline.py

=== Final staged summary ===
01_data_ingestion.ipynb                   |   1 +
 src/sales_pipeline/extract.py             | 115 ++++++++++++++++++
 src/sales_pipeline/pipeline.py            | 121 +++++++++++++++++++
 src/sales_pipeline/transform.py           | 113 ++++++++++++++++++
 src/sales_pipeline/transform_customers.py | 109 +++++++++++++++++
 src/sales_pipeline/transform_products.py  | 105 +++++++++++++++++
 src/sales_pipeline/transform_sales.py     |  87 ++++++++++++++
 src/sales_pipeline/validate.py            | 186 ++++++++++++++++++++++++++++++
 tests/test_pipeline.py                    | 104 +++++++++++++++++
 9 files changed, 941 insertions(+)


In [78]:
# Commit the initail pipeline


result = subprocess.run(
    [
        "git", "commit",
        "-m",
        "Build sales analytics data engineering pipeline"
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)


[main 6e249bb] Build sales analytics data engineering pipeline
 9 files changed, 941 insertions(+)
 create mode 100644 01_data_ingestion.ipynb
 create mode 100644 src/sales_pipeline/extract.py
 create mode 100644 src/sales_pipeline/pipeline.py
 create mode 100644 src/sales_pipeline/transform.py
 create mode 100644 src/sales_pipeline/transform_customers.py
 create mode 100644 src/sales_pipeline/transform_products.py
 create mode 100644 src/sales_pipeline/transform_sales.py
 create mode 100644 src/sales_pipeline/validate.py
 create mode 100644 tests/test_pipeline.py

Exit code: 0


In [79]:
#Varify the commit

print("Latest commit:")
print(git_command(["log", "-1", "--oneline"]))

print("\nWorking tree:")
status = git_command(["status", "--short"])
print(status if status else "Clean")


Latest commit:
6e249bb Build sales analytics data engineering pipeline

Working tree:
M 01_data_ingestion.ipynb


In [85]:

import json
from pathlib import Path

notebook_path = REPO_ROOT / "01_data_ingestion.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i in range(52, len(notebook["cells"])):
    cell = notebook["cells"][i]
    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 70}")
    print(f"CELL {i} | TYPE: {cell['cell_type']}")
    print(f"{'=' * 70}")
    print(source[:2000] if source else "[Empty source]")

    outputs = cell.get("outputs", [])
    print(f"\nSaved output count: {len(outputs)}")



CELL 52 | TYPE: code
# Restage the notebook and varify
import subprocess

def git_command(args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()

git_command(["add", "--", "01_data_ingestion.ipynb"])

print(git_command(["status", "--short"]))
print("\n=== Final staged summary ===")
print(git_command(["diff", "--cached", "--stat"]))


Saved output count: 1

CELL 53 | TYPE: code
# Commit the initail pipeline


result = subprocess.run(
    [
        "git", "commit",
        "-m",
        "Build sales analytics data engineering pipeline"
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)


Saved output count: 1

CELL 54 | TYPE: code
#Varify the commit

print("Latest commit:")
print(git_command(["log", "-1", "--oneline"]))

print("\nWorking tree:")
status =

In [86]:

import json
import shutil
from datetime import datetime
from pathlib import Path

notebook_path = REPO_ROOT / "01_data_ingestion.ipynb"

# Create a backup before making any changes
backup_path = REPO_ROOT / (
    f"01_data_ingestion_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.ipynb"
)
shutil.copy2(notebook_path, backup_path)

# Load notebook
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Cells before cleanup:", len(notebook["cells"]))

# Remove cells 52–56 (Git-management and diagnostic cells)
notebook["cells"] = notebook["cells"][:52]

# Save the cleaned notebook
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)
    f.write("\n")

print("Cells after cleanup:", len(notebook["cells"]))
print("Backup created:", backup_path.name)
print("Notebook cleanup completed.")


Cells before cleanup: 57
Cells after cleanup: 52
Backup created: 01_data_ingestion_backup_20261015_105601.ipynb
Notebook cleanup completed.


In [87]:

import subprocess
import json

notebook_path = REPO_ROOT / "01_data_ingestion.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook cell count:", len(notebook["cells"]))
print("Empty source cells:", sum(
    not "".join(cell.get("source", [])).strip()
    for cell in notebook["cells"]
))

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

print("\nGit status:")
print(result.stdout if result.stdout.strip() else "Working tree clean")


Notebook cell count: 52
Empty source cells: 0

Git status:
 M 01_data_ingestion.ipynb
?? 01_data_ingestion_backup_20261015_105601.ipynb



In [88]:
import shutil
from pathlib import Path

backup_path = REPO_ROOT / "01_data_ingestion_backup_20261015_105601.ipynb"
safe_backup_path = Path("/content") / backup_path.name

shutil.move(str(backup_path), str(safe_backup_path))

print("Backup preserved outside repository:", safe_backup_path)

Backup preserved outside repository: /content/01_data_ingestion_backup_20261015_105601.ipynb


In [89]:
import subprocess

result = subprocess.run(
    ["git", "diff", "--stat", "--", "01_data_ingestion.ipynb"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout or "No notebook changes detected.")

 01_data_ingestion.ipynb | 9799 ++++++++++++++++++++++++++++++++++++++++++++++-
 1 file changed, 9798 insertions(+), 1 deletion(-)



In [90]:

import subprocess
from pathlib import Path

notebook_path = "01_data_ingestion.ipynb"

result = subprocess.run(
    ["git", "show", f"HEAD:{notebook_path}"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

committed_text = result.stdout
current_text = (REPO_ROOT / notebook_path).read_text(encoding="utf-8")

print("Committed file size:", len(committed_text), "characters")
print("Current file size:", len(current_text), "characters")
print("Committed newline count:", committed_text.count("\n"))
print("Current newline count:", current_text.count("\n"))
print("Committed starts with:", repr(committed_text[:80]))
print("Current starts with:", repr(current_text[:80]))


Committed file size: 339326 characters
Current file size: 407571 characters
Committed newline count: 0
Current newline count: 9798
Committed starts with: '{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"authorshi'
Current starts with: '{\n "nbformat": 4,\n "nbformat_minor": 0,\n "metadata": {\n  "colab": {\n   "provenan'


In [92]:

import json
import shutil
from pathlib import Path

notebook_path = REPO_ROOT / "01_data_ingestion.ipynb"

# Preserve the current cleaned notebook before reformatting
backup_path = Path("/content/01_data_ingestion_before_compact.json.ipynb")
shutil.copy2(notebook_path, backup_path)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

assert len(notebook["cells"]) == 52, (
    f"Expected 52 cells, found {len(notebook['cells'])}"
)

# Write compact JSON, matching the original file's format
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, separators=(",", ":"))

print("Notebook cells:", len(notebook["cells"]))
print("Compact formatting restored.")
print("Backup:", backup_path)


Notebook cells: 52
Compact formatting restored.
Backup: /content/01_data_ingestion_before_compact.json.ipynb


In [96]:

import subprocess

def run_git(args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )

# Stage only the verified notebook cleanup
run_git(["add", "--", "01_data_ingestion.ipynb"])

# Commit the cleanup
result = run_git([
    "commit",
    "-m",
    "Clean up data ingestion notebook",
])

print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

print("\nLatest commit:")
print(run_git(["log", "-1", "--oneline"]).stdout)

print("Working tree status:")
status = run_git(["status", "--short"]).stdout.strip()
print(status if status else "Clean")


[main a4b6dba] Clean up data ingestion notebook
 1 file changed, 1 insertion(+), 1 deletion(-)


Latest commit:
a4b6dba Clean up data ingestion notebook

Working tree status:
Clean


In [99]:
# Varify Pipeline
import sys
from pathlib import Path

sys.path.insert(0, str(REPO_ROOT / "src"))

from sales_pipeline.pipeline import run_pipeline

DATA_DIR = REPO_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

summary = run_pipeline(DATA_DIR, PROCESSED_DIR)
print("\nPIPELINE SUMMARY")
print(summary)


SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...
SALES ANALYTICS PIPELINE — DATA QUALITY REPORT
[PASS] Customers: customer_id has no nulls
        Missing values: 0
[PASS] Customers: customer_id is unique
        Duplicate rows for key: 0
[PASS] Orders: order_id has no nulls
        Missing values: 0
[PASS] Orders: order_id is unique
        Duplicate rows for key: 0
[PASS] Order items: order_id has no nulls
        Missing values: 0
[PASS] Order items: order_item_id has no nulls
        Missing values: 0
[PASS] Order items: order_id, order_item_id is unique
        Duplicate rows for key: 0
[PASS] Products: product_id has no nulls
        Missing values: 0
[PASS] Products: product_id is unique
        Duplicate rows for key: 0
[PASS] Sellers: seller_id has no nulls
        Missing values: 0
[PASS] Sellers: seller_id is unique
        Duplicate rows for key: 0
[PASS] Orders: customer_id has no nulls
        Missing values: 0
[PASS] Order items: product_id has no nulls
   

In [100]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-v",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print("--- TEST RESULTS ---")
print(result.stdout)
print(result.stderr)

print("\nTest exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Automated tests failed. Review the test output above."
    )


--- TEST RESULTS ---

test_category_names_are_unique (test_pipeline.TestPipelineOutputs.test_category_names_are_unique) ... ok
test_customer_ids_are_unique (test_pipeline.TestPipelineOutputs.test_customer_ids_are_unique) ... ok
test_customer_repeat_flag (test_pipeline.TestPipelineOutputs.test_customer_repeat_flag) ... ok
test_expected_outputs_exist (test_pipeline.TestPipelineOutputs.test_expected_outputs_exist) ... ok
test_monthly_order_month_is_unique (test_pipeline.TestPipelineOutputs.test_monthly_order_month_is_unique) ... ok
test_order_ids_are_unique (test_pipeline.TestPipelineOutputs.test_order_ids_are_unique) ... ok
test_order_status_counts_reconcile (test_pipeline.TestPipelineOutputs.test_order_status_counts_reconcile) ... ok
test_product_and_category_revenue_reconcile (test_pipeline.TestPipelineOutputs.test_product_and_category_revenue_reconcile) ... ok
test_product_and_category_units_reconcile (test_pipeline.TestPipelineOutputs.test_product_and_category_units_reconcile) ... ok